# Team 3: Spark Analytics
Full-dataset PySpark and Spark SQL analysis for Owl Analytics.


In [3]:
!pip -q install pyspark


In [4]:
from pyspark.sql import SparkSession, functions as F, types as T
spark=SparkSession.builder.appName('OwlAnalytics').getOrCreate()
print('Spark session started')


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
path='data/clean/cleaned_market_data.csv'
df=(spark.read.option('header',True).option('inferSchema',True).csv(path)
    .withColumn('open_time',F.to_timestamp('open_time'))
    .withColumn('close_time',F.to_timestamp('close_time')))
print('Row count:',df.count()); print('Columns:',df.columns); df.printSchema(); df.show(5,truncate=False)


In [ ]:
df=(df.withColumn('price_range',F.col('high')-F.col('low'))
 .withColumn('price_change',F.col('close')-F.col('open'))
 .withColumn('percent_change',(F.col('close')-F.col('open'))/F.col('open')*100)
 .withColumn('candle_direction',F.when(F.col('close')>F.col('open'),'up').when(F.col('close')<F.col('open'),'down').otherwise('flat'))
 .withColumn('hour',F.hour('open_time')).withColumn('day_of_week',F.date_format('open_time','E'))
 .withColumn('date',F.to_date('open_time')))
df.createOrReplaceTempView('market_data'); print('Temporary SQL view created: market_data'); spark.sql('SELECT * FROM market_data LIMIT 10').show()


## 1. Records and price overview by symbol


In [ ]:
q="""SELECT symbol, COUNT(*) records, ROUND(AVG(close),6) avg_close, ROUND(MIN(low),6) min_low, ROUND(MAX(high),6) max_high FROM market_data GROUP BY symbol ORDER BY symbol"""
spark.sql(q).show(100,truncate=False)


## 2. Volatility ranking


In [ ]:
q="""SELECT symbol, ROUND(AVG(price_range),6) avg_price_range, ROUND(STDDEV(percent_change),6) percent_change_stddev FROM market_data GROUP BY symbol ORDER BY percent_change_stddev DESC"""
spark.sql(q).show(100,truncate=False)


## 3. Trading activity ranking


In [ ]:
q="""SELECT symbol, ROUND(SUM(volume),2) total_volume, SUM(trade_count) total_trades, ROUND(AVG(trade_count),2) avg_trades FROM market_data GROUP BY symbol ORDER BY total_trades DESC"""
spark.sql(q).show(100,truncate=False)


## 4. Candle direction counts


In [ ]:
q="""SELECT symbol, candle_direction, COUNT(*) candles FROM market_data GROUP BY symbol,candle_direction ORDER BY symbol,candles DESC"""
spark.sql(q).show(100,truncate=False)


## 5. Strongest positive and negative hourly moves


In [ ]:
q="""SELECT symbol, open_time, ROUND(percent_change,4) percent_change, candle_direction FROM market_data ORDER BY ABS(percent_change) DESC LIMIT 20"""
spark.sql(q).show(100,truncate=False)


## 6. Activity by hour of day


In [ ]:
q="""SELECT hour, SUM(trade_count) total_trades, ROUND(SUM(volume),2) total_volume FROM market_data GROUP BY hour ORDER BY total_trades DESC"""
spark.sql(q).show(100,truncate=False)


## 7. Activity by day of week


In [ ]:
q="""SELECT day_of_week, SUM(trade_count) total_trades, ROUND(AVG(ABS(percent_change)),4) avg_absolute_change FROM market_data GROUP BY day_of_week ORDER BY total_trades DESC"""
spark.sql(q).show(100,truncate=False)


## 8. Daily market movement


In [ ]:
q="""SELECT date, ROUND(AVG(percent_change),4) avg_percent_change, ROUND(SUM(volume),2) total_volume FROM market_data GROUP BY date ORDER BY date"""
spark.sql(q).show(100,truncate=False)


In [ ]:
summary=spark.sql("""SELECT symbol, COUNT(*) records, ROUND(AVG(close),6) average_close, ROUND(AVG(price_range),6) average_price_range, ROUND(STDDEV(percent_change),6) volatility_score, SUM(trade_count) total_trades, ROUND(SUM(volume),2) total_volume, ROUND(AVG(percent_change),6) average_percent_change FROM market_data GROUP BY symbol ORDER BY volatility_score DESC""")
summary.show(truncate=False)
summary.coalesce(1).write.mode('overwrite').option('header',True).csv('results/spark_market_summary_output')
summary.toPandas().to_csv('results/spark_market_summary.csv',index=False)
print('Saved results/spark_market_summary.csv')


## Interpretation
The final ranking separates volatility from activity. A market can have a high trading count without having the largest percentage swings. The hourly and weekday queries show when the dataset was most active, while the candle-direction and extreme-move queries identify directional behaviour and outliers. Exact rankings depend on the API extraction time, so the saved CSV is the authoritative run output.
